In [6]:
import pickle
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from scipy import sparse
from scipy.sparse import diags
from sklearn.preprocessing import MinMaxScaler
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm import LightFM
from lightfm.evaluation import precision_at_k
from sklearn.feature_extraction.text import TfidfVectorizer
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score, reciprocal_rank

# Model Loading

In [7]:
with open('anime_recsys_model.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

model = loaded_data['model']
dataset = loaded_data['dataset']
item_features = loaded_data['item_features']
anime_df = loaded_data['anime_df']

# Model Test

In [17]:
anime_df = pd.read_csv('clear_anime.csv')
anime_df1 = pd.read_csv('anime.csv')
merged_df = pd.merge(anime_df, anime_df1, on='MAL_ID', how='inner')
merged_df=merged_df.rename(columns={'MAL_ID': 'anime_id'})
anime_df = merged_df[['anime_id', 'Name','English name']]
anime_df

,anime_id,Name,English name
0,1,Cowboy Bebop,Cowboy Bebop
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop:The Movie
2,6,Trigun,Trigun
3,7,Witch Hunter Robin,Witch Hunter Robin
4,8,Bouken Ou Beet,Beet the Vandel Buster
...,...,...,...
17458,48481,Daomu Biji Zhi Qinling Shen Shu,Unknown
17459,48483,Mieruko-chan,Unknown
17460,48488,Higurashi no Naku Koro ni Sotsu,Higurashi:When They Cry – SOTSU
17461,48491,Yama no Susume: Next Summit,Unknown


## Users params

In [45]:
ratings = {
    "Monster": 10,                
    "Kenpuu Denki Berserk": 10,                 
    "Mushishi": 9,                
    "Vinland Saga": 9,             
    "Odd Taxi": 10,                
    "Perfect Blue": 10,            
    "Cowboy Bebop": 9,            
    "Shingeki no Kyojin": 7,          
    "Kimetsu no Yaiba": 6, 
    "Sword Art Online": 2,        
    "Kanojo, Okarishimasu": 1,        
    "Fairy Tail": 3,              
    "High School DxD": 2
}

In [37]:
def get_recommendations(user_ratings, model, dataset, anime_df, item_features, num_recommendations=10):
    mid_map = dataset.mapping()[2]
    inverse_map = {v: k for k, v in mid_map.items()}
    name_to_real_id = dict(zip(anime_df['Name'], anime_df['anime_id']))
    item_biases, item_representations = model.get_item_representations(features=item_features)
    user_vector = np.zeros(item_representations.shape[1])
    seen_ids = []

    for name, rating in user_ratings.items():
        if name in name_to_real_id:
            real_id = name_to_real_id[name]
            if real_id in mid_map:
                internal_id = mid_map[real_id]
                seen_ids.append(internal_id)
                weight = rating - 5.0
                user_vector += item_representations[internal_id] * weight
        else:
            print(f"Anime '{name}' not found in dataset.")

    scores = item_representations.dot(user_vector) + item_biases
    scores[seen_ids] = -np.inf

    top_indices = np.argsort(-scores)[:num_recommendations]
    top_real_ids = [inverse_map[i] for i in top_indices]

    res_df = pd.DataFrame({
        'anime_id': top_real_ids,
    })
    
    final_df = res_df.merge(anime_df[['anime_id', 'Name', 'English name']], on='anime_id', how='left')
    return final_df[['Name', 'English name']]

In [46]:
get_recommendations(ratings, model, dataset, anime_df, item_features, num_recommendations=10)

,Name,English name
0,Rainbow: Nisha Rokubou no Shichinin,Rainbow
1,Aoi Bungaku Series,Unknown
2,Ginga Eiyuu Densetsu,Legend of the Galactic Heroes
3,Memories,Unknown
4,Serial Experiments Lain,Serial Experiments Lain
5,Gankutsuou,Gankutsuou:The Count of Monte Cristo
6,Texhnolyze,Texhnolyze
7,Ergo Proxy,Ergo Proxy
8,Mononoke,Mononoke
9,Paprika,Paprika
